 # Dialogue-level BLEU on symbolic annotation tokens (MultiWOZ vs LLM annotations)

 We compare LLM-generated MultiWOZ-style annotations against the original MultiWOZ labels.
 Because the data is symbolic (dialog acts + slot/value pairs), we:
 1) linearize each turn into a canonical representation
 2) tokenize into symbols (acts and slot=value)
 3) concatenate turns per dialogue, inserting <TURN> boundaries
 4) compute corpus BLEU-1 and BLEU-4 using SacreBLEU

In [1]:
import json
import re
from collections import defaultdict
from json import JSONDecodeError
from sacrebleu.metrics import BLEU

In [2]:
from pathlib import Path

from collections import defaultdict

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"
RES_DIR = Path("../results")
RES_DIR.mkdir(parents=True, exist_ok=True)

DIALOGS_PATH = RAW_DIR / "multiwoz_original.json"
ACTS_PATH = RAW_DIR / "dialogue_acts.json"
ANN1_PATH = PROC_DIR / "ann1.txt"
COMMON_IDS_PATH = PROC_DIR / "common_dialog_ids.json"

print("DIALOGS:", DIALOGS_PATH.resolve(), DIALOGS_PATH.exists())
print("ACTS:   ", ACTS_PATH.resolve(), ACTS_PATH.exists())
print("ANN1:   ", ANN1_PATH.resolve(), ANN1_PATH.exists())
print("COMMON: ", COMMON_IDS_PATH.resolve(), COMMON_IDS_PATH.exists())



DIALOGS: C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\data\raw\multiwoz_original.json True
ACTS:    C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\data\raw\dialogue_acts.json True
ANN1:    C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\data\processed\ann1.txt True
COMMON:  C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\data\processed\common_dialog_ids.json True


In [3]:
with COMMON_IDS_PATH.open("r", encoding="utf-8") as f:
    common_ids = set(json.load(f))

with DIALOGS_PATH.open("r", encoding="utf-8") as f:
    dialogs = json.load(f)

with ACTS_PATH.open("r", encoding="utf-8") as f:
    orig_acts = json.load(f)

print("Common IDs:", len(common_ids))
print("Dialogs loaded:", len(dialogs))
print("Dialogue acts loaded:", len(orig_acts))


Common IDs: 10438
Dialogs loaded: 10438
Dialogue acts loaded: 10438


In [4]:
def parse_annotation_txt(path):
    """Parse the custom TXT annotation export into a MultiWOZ-like dict.

    Each line:
        <utterance>\t<json1>[\t<json2>\t<json3>...]

    Returns:
        dict[dialog_id] -> {"log_by_turn": dict[int, turn_obj]}
    """
    dialogs = defaultdict(lambda: {"log_by_turn": {}})

    stats = {
        "lines_total": 0,
        "lines_no_json": 0,
        "lines_no_base": 0,
        "lines_no_da": 0,
        "lines_parsed": 0,
        "turn_overwrites": 0,
    }

    with open(path, encoding="utf-8") as f:
        for lineno, raw_line in enumerate(f, start=1):
            stats["lines_total"] += 1

            line = raw_line.rstrip("\n")
            if not line.strip():
                continue

            parts = line.split("\t")
            if len(parts) < 2:
                stats["lines_no_json"] += 1
                continue

            utterance = parts[0]

            json_strs = []
            for p in parts[1:]:
                p = p.strip()
                if not p:
                    continue
                if "{" in p and "}" in p:
                    start = p.index("{")
                    end = p.rfind("}") + 1
                    json_strs.append(p[start:end])

            if not json_strs:
                stats["lines_no_json"] += 1
                continue

            objs = []
            for js in json_strs:
                try:
                    objs.append(json.loads(js))
                except JSONDecodeError as e:
                    print(f"[ERROR] JSON parse failed on line {lineno}: {e}")
                    print("  Chunk:", repr(js))
                    raise

            base = None
            for o in objs:
                if "_dialog_index" in o and "_turn_index" in o:
                    base = o
                    break
            if base is None:
                stats["lines_no_base"] += 1
                continue

            dialog_id = base["_dialog_index"]
            t_idx = int(base["_turn_index"])

            speaker = next((o.get("speaker") for o in objs if "speaker" in o), "unknown")

            dialog_act = {}
            for obj in objs:
                act_name = obj.get("dialog_act")
                if not act_name:
                    continue
                slots = obj.get("slots", {}) or {}

                # IMPORTANT: keep raw values (do NOT str() here) so strict/normalized stay meaningful
                slot_list = [[k, v] for k, v in slots.items()]

                dialog_act.setdefault(act_name, []).extend(slot_list)

            if not dialog_act:
                stats["lines_no_da"] += 1
                continue

            turn_obj = {"text": utterance, "dialog_act": dialog_act, "speaker": speaker}

            if t_idx in dialogs[dialog_id]["log_by_turn"]:
                stats["turn_overwrites"] += 1

            dialogs[dialog_id]["log_by_turn"][t_idx] = turn_obj
            stats["lines_parsed"] += 1

    print("parse_annotation_txt stats:", stats)
    return dict(dialogs)

cand_data = parse_annotation_txt(ANN1_PATH)
print("Loaded candidate dialogs:", len(cand_data))


parse_annotation_txt stats: {'lines_total': 144441, 'lines_no_json': 1404, 'lines_no_base': 0, 'lines_no_da': 34, 'lines_parsed': 143000, 'turn_overwrites': 0}
Loaded candidate dialogs: 10438


In [5]:
def normalize_slot_value(slot, slot_val, mode="strict"):
    """Normalize a slot value according to mode.

    :param slot: normalized slot name (lowercase)
    :param slot_val: raw slot value (any type)
    :param mode: "strict" | "normalized" | "lenient"
    :return: normalized string (possibly empty)
    """
    if mode == "strict":
        return "None" if slot_val is None else str(slot_val).strip()

    sval = "" if slot_val is None else str(slot_val).strip().lower()

    if sval in {"none", "null", "nan", "n/a", "?"}:
        sval = ""

    if slot in {"wifi", "internet", "parking"}:
        truthy = {"true", "yes", "y", "1"}
        falsy = {"false", "no", "n", "0"}
        if mode == "lenient":
            truthy |= {"free", "available"}

        if sval in truthy:
            sval = "yes"
        elif sval in falsy:
            sval = "no"

    return sval

In [6]:
def linearize_turn(turn_obj, mode="strict"):
    """Linearize one turn into a canonical string: 'act1,act2(slot=a,slot=b)'.

    - acts kept in insertion order
    - all slot/value pairs collected across acts
    - slot pairs sorted alphabetically (preserve duplicates)
    - empty values kept as 'slot='

    :param turn_obj: dict with "dialog_act"
    :param mode: "strict" | "normalized" | "lenient"
    :return: canonical string
    """
    da = turn_obj.get("dialog_act", {}) or {}
    act_names = list(da.keys())  # insertion order

    if mode not in {"strict", "normalized", "lenient"}:
        raise ValueError(f"Unknown mode: {mode}")

    slot_pairs = []
    for act_name in act_names:
        for slot_name, slot_val in da.get(act_name, []) or []:
            slot = str(slot_name).strip().lower()
            sval = normalize_slot_value(slot, slot_val, mode=mode)
            slot_pairs.append((slot, sval))

    acts_part = ",".join(str(a).strip().lower() for a in act_names)

    slot_pairs.sort(key=lambda x: (x[0], x[1]))
    slots_str = ",".join(f"{k}={v}" for k, v in slot_pairs) if slot_pairs else ""

    return f"{acts_part}({slots_str})"

In [7]:
def bleu_tokenize(linear_str):
    """Tokenize a linearized turn string into BLEU tokens (acts + slot=value)."""
    s = linear_str.strip()

    if "(" in s and ")" in s:
        acts_part, slots_part = s.split("(", 1)
        slots_part = slots_part.rsplit(")", 1)[0]
    else:
        acts_part, slots_part = s, ""

    act_tokens = [t.strip() for t in acts_part.split(",") if t.strip()]
    slot_tokens = [t.strip() for t in slots_part.split(",") if t.strip()]

    tokens = [re.sub(r"\s+", "", t) for t in (act_tokens + slot_tokens)]
    return [t for t in tokens if t]

In [8]:
def build_dialog_token_sequences(data, mode="strict", add_turn_boundary=True):
    """Build dict[dialog_id] -> flattened token sequence for BLEU.

    Supports:
      - original MultiWOZ: {"log": [turn0, turn1, ...]}
      - candidate: {"log_by_turn": {idx: turn_obj, ...}}
    """
    out = {}

    for d_id, dialog in data.items():
        tokens = []

        if dialog.get("log_by_turn"):
            for t_idx in sorted(dialog["log_by_turn"].keys()):
                turn = dialog["log_by_turn"][t_idx]
                linear = linearize_turn(turn, mode=mode)
                tokens.extend(bleu_tokenize(linear))
                if add_turn_boundary:
                    tokens.append("<TURN>")
        else:
            log = dialog.get("log", []) or []
            for turn in log:
                linear = linearize_turn(turn, mode=mode)
                tokens.extend(bleu_tokenize(linear))
                if add_turn_boundary:
                    tokens.append("<TURN>")

        out[d_id] = tokens

    return out

In [9]:
bleu1_metric = BLEU(max_ngram_order=1, effective_order=True)
bleu4_metric = BLEU(max_ngram_order=4, effective_order=True)

def compute_bleu_scores(data_ref, data_hyp, mode="strict"):
    """Compute corpus BLEU-1 and BLEU-4 for a given mode."""
    ref_dialogs = build_dialog_token_sequences(data_ref, mode=mode, add_turn_boundary=True)
    hyp_dialogs = build_dialog_token_sequences(data_hyp, mode=mode, add_turn_boundary=True)

    common_ids = sorted(set(ref_dialogs) & set(hyp_dialogs))

    refs, hyps = [], []
    for d in common_ids:
        r = " ".join(ref_dialogs[d]).strip()
        h = " ".join(hyp_dialogs[d]).strip()
        if not r or not h:
            continue
        refs.append(r)
        hyps.append(h)

    return {
        "mode": mode,
        "dialogs_used": len(refs),
        "bleu1": bleu1_metric.corpus_score(hyps, [refs]).score,
        "bleu4": bleu4_metric.corpus_score(hyps, [refs]).score,
        "turn_boundary": True,
    }

In [10]:
for m in ["strict", "normalized", "lenient"]:
    print(m, compute_bleu_scores(dialogs, cand_data, mode=m))

strict {'mode': 'strict', 'dialogs_used': 10438, 'bleu1': 63.806533248061, 'bleu4': 36.6620245783474, 'turn_boundary': True}
normalized {'mode': 'normalized', 'dialogs_used': 10438, 'bleu1': 70.17048177913554, 'bleu4': 42.21928198648245, 'turn_boundary': True}
lenient {'mode': 'lenient', 'dialogs_used': 10438, 'bleu1': 70.32382307612647, 'bleu4': 42.51524818203763, 'turn_boundary': True}


In [11]:
# Sanity check: show one dialog token stream head
ref_strict = build_dialog_token_sequences(dialogs, mode="strict", add_turn_boundary=True)
hyp_strict = build_dialog_token_sequences(cand_data, mode="strict", add_turn_boundary=True)

common_ids = sorted(set(ref_strict) & set(hyp_strict))
sample_id = common_ids[0]

print("Sample dialog:", sample_id)
print("REF (first 40):", ref_strict[sample_id][:40])
print("HYP (first 40):", hyp_strict[sample_id][:40])

Sample dialog: MUL0001.json
REF (first 40): ['restaurant-inform', 'food=indian', '<TURN>', 'restaurant-request', 'restaurant-inform', 'area=?', 'choice=many', 'food=Indian', 'price=thatpricerange', '<TURN>', 'restaurant-inform', 'area=centre', 'food=indian', 'price=expensive', '<TURN>', 'booking-inform', 'restaurant-recommend', 'area=center', 'food=Indian', 'name=SaffronBrasserie', 'none=none', 'price=expensive', '<TURN>', 'restaurant-inform', 'day=saturday', 'people=6', 'time=19:30', '<TURN>', 'booking-book', 'ref=RF00JUFQ', '<TURN>', 'hotel-inform', 'internet=no', 'stars=3', 'type=hotel', '<TURN>', 'booking-inform', 'hotel-inform', 'area=north', 'internet=none']
HYP (first 40): ['restaurant-request', 'food=indian', '<TURN>', 'restaurant-inform', 'restaurant-request', 'food=indian', 'price=', '<TURN>', 'restaurant-request', 'area=centre', 'food=indian', 'price=expensive', '<TURN>', 'restaurant-recommend', 'booking-offerbook', 'area=centre', 'food=indian', 'price=expensive', '<TURN>', 

In [12]:
# --- BLEU-only results dump (matches what you print above) ---
import json

bleu_results = {
    "strict": compute_bleu_scores(dialogs, cand_data, mode="strict"),
    "normalized": compute_bleu_scores(dialogs, cand_data, mode="normalized"),
    "lenient": compute_bleu_scores(dialogs, cand_data, mode="lenient"),
}

OUT_PATH = RES_DIR / "bleu_ann1_vs_original.json"
with OUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(bleu_results, f, indent=2)

print("Saved:", OUT_PATH.resolve())
print(json.dumps(bleu_results, indent=2))



Saved: C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\results\bleu_ann1_vs_original.json
{
  "strict": {
    "mode": "strict",
    "dialogs_used": 10438,
    "bleu1": 63.806533248061,
    "bleu4": 36.6620245783474,
    "turn_boundary": true
  },
  "normalized": {
    "mode": "normalized",
    "dialogs_used": 10438,
    "bleu1": 70.17048177913554,
    "bleu4": 42.21928198648245,
    "turn_boundary": true
  },
  "lenient": {
    "mode": "lenient",
    "dialogs_used": 10438,
    "bleu1": 70.32382307612647,
    "bleu4": 42.51524818203763,
    "turn_boundary": true
  }
}


In [ ]:

from pathlib import Path

# Assumes these already exist from your notebook:
# dialogs, cand_data, build_dialog_token_sequences

def jaccard_similarity(tokens_a, tokens_b):
    """Compute Jaccard similarity between two token sets."""
    set_a = set(tokens_a)
    set_b = set(tokens_b)
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

rows = []

for mode in ["strict", "normalized", "lenient"]:
    ref_dialogs = build_dialog_token_sequences(dialogs, mode=mode, add_turn_boundary=True)
    hyp_dialogs = build_dialog_token_sequences(cand_data, mode=mode, add_turn_boundary=True)

    common_ids = sorted(set(ref_dialogs) & set(hyp_dialogs))

    for dialog_id in common_ids:
        ref_tokens = ref_dialogs[dialog_id]
        hyp_tokens = hyp_dialogs[dialog_id]

        rows.append({
            "dialog_id": dialog_id,
            "mode": mode,
            "ref_len": len(ref_tokens),
            "hyp_len": len(hyp_tokens),
            "jaccard": jaccard_similarity(ref_tokens, hyp_tokens),
            "ref_token_str": " ".join(ref_tokens),
            "hyp_token_str": " ".join(hyp_tokens),
        })

df_dialog = pd.DataFrame(rows)

print("Shape:", df_dialog.shape)
print("\nColumns:")
print(df_dialog.columns.tolist())

print("\nFirst 5 rows:")
print(df_dialog.head())

OUT_PATH = Path("../results/dialog_level_overlap.csv")
df_dialog.to_csv(OUT_PATH, index=False, encoding="utf-8")
print("\nSaved:", OUT_PATH.resolve())

Shape: (31314, 7)

Columns:
['dialog_id', 'mode', 'ref_len', 'hyp_len', 'jaccard', 'ref_token_str', 'hyp_token_str']

First 5 rows:
      dialog_id    mode  ref_len  hyp_len   jaccard  \
0  MUL0001.json  strict       88       79  0.365385   
1  MUL0002.json  strict       62       52  0.289474   
2  MUL0003.json  strict       90       66  0.440000   
3  MUL0004.json  strict       79       58  0.352941   
4  MUL0005.json  strict       92       76  0.296875   

                                       ref_token_str  \
0  restaurant-inform food=indian <TURN> restauran...   
1  restaurant-inform area=center none=none <TURN>...   
2  hotel-inform internet=yes type=guesthouse <TUR...   
3  restaurant-inform area=Centre name=nandoscityc...   
4  restaurant-inform food=seafood price=expensive...   

                                       hyp_token_str  
0  restaurant-request food=indian <TURN> restaura...  
1  restaurant-request area=citycenter <TURN> rest...  
2  hotel-request type=guesthouse wi

In [ ]:
def jaccard_similarity(tokens_a, tokens_b):
    set_a = set(tokens_a)
    set_b = set(tokens_b)
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

rows = []

for add_turn_boundary in [True, False]:
    label = "with_turn" if add_turn_boundary else "without_turn"

    for mode in ["strict", "normalized", "lenient"]:
        ref_dialogs = build_dialog_token_sequences(dialogs, mode=mode, add_turn_boundary=add_turn_boundary)
        hyp_dialogs = build_dialog_token_sequences(cand_data, mode=mode, add_turn_boundary=add_turn_boundary)

        common_ids = sorted(set(ref_dialogs) & set(hyp_dialogs))

        for dialog_id in common_ids:
            ref_tokens = ref_dialogs[dialog_id]
            hyp_tokens = hyp_dialogs[dialog_id]

            rows.append({
                "dialog_id": dialog_id,
                "mode": mode,
                "boundary_setting": label,
                "jaccard": jaccard_similarity(ref_tokens, hyp_tokens),
            })

df_check = pd.DataFrame(rows)

summary = (
    df_check.groupby(["boundary_setting", "mode"])["jaccard"]
    .agg(["mean", "median", "std"])
    .reset_index()
)

print(summary)

  boundary_setting        mode      mean    median       std
0        with_turn     lenient  0.408374  0.400000  0.087659
1        with_turn  normalized  0.403591  0.395833  0.089096
2        with_turn      strict  0.355055  0.346939  0.080357
3     without_turn     lenient  0.389295  0.382979  0.088762
4     without_turn  normalized  0.384467  0.377358  0.090037
5     without_turn      strict  0.335494  0.333333  0.080109


In [17]:
from sacrebleu.metrics import BLEU
import pandas as pd

bleu = BLEU(effective_order=True)

rows = []

for add_turn_boundary in [True, False]:
    boundary_setting = "with_turn" if add_turn_boundary else "without_turn"

    for mode in ["strict", "normalized", "lenient"]:
        ref_dialogs = build_dialog_token_sequences(
            dialogs,
            mode=mode,
            add_turn_boundary=add_turn_boundary
        )
        hyp_dialogs = build_dialog_token_sequences(
            cand_data,
            mode=mode,
            add_turn_boundary=add_turn_boundary
        )

        common_ids = sorted(set(ref_dialogs) & set(hyp_dialogs))

        refs = [" ".join(ref_dialogs[d]) for d in common_ids]
        hyps = [" ".join(hyp_dialogs[d]) for d in common_ids]

        score = bleu.corpus_score(hyps, [refs])

        rows.append({
            "boundary_setting": boundary_setting,
            "mode": mode,
            "bleu": score.score
        })

df_bleu_check = pd.DataFrame(rows)
print(df_bleu_check)

  boundary_setting        mode       bleu
0        with_turn      strict  36.662025
1        with_turn  normalized  42.219282
2        with_turn     lenient  42.515248
3     without_turn      strict  21.936731
4     without_turn  normalized  27.110106
5     without_turn     lenient  27.477412


In [18]:
from sacrebleu.metrics import BLEU
import pandas as pd
from pathlib import Path

bleu = BLEU(effective_order=True)

def jaccard_similarity(tokens_a, tokens_b):
    set_a = set(tokens_a)
    set_b = set(tokens_b)
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

rows = []

for mode in ["strict", "normalized", "lenient"]:
    ref_dialogs = build_dialog_token_sequences(dialogs, mode=mode, add_turn_boundary=False)
    hyp_dialogs = build_dialog_token_sequences(cand_data, mode=mode, add_turn_boundary=False)

    common_ids = sorted(set(ref_dialogs) & set(hyp_dialogs))

    for dialog_id in common_ids:
        ref_tokens = ref_dialogs[dialog_id]
        hyp_tokens = hyp_dialogs[dialog_id]

        ref_text = " ".join(ref_tokens)
        hyp_text = " ".join(hyp_tokens)

        bleu_score = bleu.corpus_score([hyp_text], [[ref_text]]).score

        rows.append({
            "dialog_id": dialog_id,
            "mode": mode,
            "ref_len": len(ref_tokens),
            "hyp_len": len(hyp_tokens),
            "jaccard": jaccard_similarity(ref_tokens, hyp_tokens),
            "bleu_dialog": bleu_score,
        })

df_no_turn = pd.DataFrame(rows)

OUT_PATH = Path("dialog_level_overlap_no_turn.csv")
df_no_turn.to_csv(OUT_PATH, index=False, encoding="utf-8")

print("Shape:", df_no_turn.shape)
print("\nColumns:")
print(df_no_turn.columns.tolist())
print("\nFirst 10 rows:")
print(df_no_turn.head(10))
print("\nMean scores by mode:")
print(df_no_turn.groupby("mode")[["jaccard", "bleu_dialog"]].mean())
print("\nSaved:", OUT_PATH.resolve())

Shape: (31314, 6)

Columns:
['dialog_id', 'mode', 'ref_len', 'hyp_len', 'jaccard', 'bleu_dialog']

First 10 rows:
      dialog_id    mode  ref_len  hyp_len   jaccard  bleu_dialog
0  MUL0001.json  strict       68       59  0.352941    22.116362
1  MUL0002.json  strict       48       38  0.270270    14.266022
2  MUL0003.json  strict       74       50  0.428571    20.273414
3  MUL0004.json  strict       63       42  0.340000    14.898419
4  MUL0005.json  strict       74       58  0.285714    20.457801
5  MUL0006.json  strict       69       51  0.327273    16.054153
6  MUL0007.json  strict       51       40  0.333333    18.365598
7  MUL0008.json  strict       98       69  0.338462    14.311300
8  MUL0009.json  strict       53       46  0.317073    22.671846
9  MUL0010.json  strict       80       69  0.241379    24.573123

Mean scores by mode:
             jaccard  bleu_dialog
mode                             
lenient     0.389295    26.198219
normalized  0.384467    25.862181
strict      0